In [13]:
import os
import finaletoolkit
from tqdm import tqdm
import subprocess
import glob
import numpy as np

In [14]:
def read_ctcf_create_intervals(path):
    with open(path, 'rt') as ctcf:
        for line in tqdm(ctcf):
            intervals = []
            chrom, start, end, _, _, strand = line.strip().split('\t')
            pos = (int(start) + int(end)) //2
            if strand=="+":
                interval_range = range(pos-2000, pos+2001)
            elif strand=="-":
                interval_range = reversed(range(pos-2000, pos+2001))
            else:
                raise ValueError(f"There is an error in the formatting of your CTCF data. Expected either '+' or '-' for the strand. Got {strand}")
            for interval_start in interval_range:
                intervals.append([chrom[3:], interval_start, interval_start+1])
            filename = f"./tmp/{chrom}_{start}_{end}.ctcf.intervals.bed"
            with open(filename, 'w') as output:
                for interval in intervals:
                    output.write(f"{interval[0]}\t{interval[1]}\t{interval[2]}\n")

In [15]:
read_ctcf_create_intervals("../data/ctcf.hg19.bed")

8719it [00:41, 209.76it/s]


In [16]:
with open("../data/ctcf.hg19.bed", 'rt') as ctcf, open('ctcf.commands.txt', 'w') as command_file:
    for line in tqdm(ctcf):
        chrom, start, end, _, _, strand = line.strip().split('\t')
        filename = f"/jet/home/rbandaru/ravi/finaletoolkit_figures/2B/tmp/{chrom}_{start}_{end}.ctcf.intervals.bed"
        out_filename = f"/jet/home/rbandaru/ravi/finaletoolkit_figures/2B/tmp/{chrom}_{start}_{end}.ctcf.intervals.wcov.bed"
        command = [
            '/jet/home/rbandaru/ravi/.conda/envs/ravi-ftk/bin/finaletools', 'coverage-intervals', 
            '-o', out_filename, 
            '-q', '30',
            '-w', '1',
            '-i', 'any',
            '/jet/home/rbandaru/ravi/finaletoolkit_figures/data/BH01.hg19.mdups.bam', 
            filename
        ]
        command_str = ' '.join(command)
        command_file.write(command_str + '\n')

8719it [00:00, 657854.59it/s]


In [17]:
from tqdm import tqdm

# Open the input file containing commands and the output shell script
with open('ctcf.commands.txt', 'r') as command_file, open('submit_ctcf_jobs.sh', 'w') as sbatch_script:
    # Write a shebang to make the script executable (optional, for direct execution)
    sbatch_script.write("#!/bin/bash\n\n")
    
    # Buffer to store commands in chunks of 10
    command_buffer = []
    
    # Process each line in the command file
    for line in tqdm(command_file):
        command = line.strip()
        command_buffer.append(command)
        
        # When buffer reaches 10 commands, create an sbatch command
        if len(command_buffer) == 10:
            combined_commands = "; ".join(command_buffer)
            sbatch_command = f"sbatch -p RM-shared -t 3:00:00 --wrap='{combined_commands}'\n"
            sbatch_script.write(sbatch_command)
            command_buffer = []  # Clear the buffer

    # Handle any remaining commands (less than 10)
    if command_buffer:
        combined_commands = "; ".join(command_buffer)
        sbatch_command = f"sbatch -p RM-shared -t 3:00:00 --wrap='{combined_commands}'\n"
        sbatch_script.write(sbatch_command)

# Make the script executable (if running from a Python script)
import os
os.chmod('submit_ctcf_jobs.sh', 0o755)

print("Job submission script 'submit_ctcf_jobs.sh' created successfully.")


8719it [00:00, 1143138.28it/s]

Job submission script 'submit_ctcf_jobs.sh' created successfully.


In [20]:
directory = "/jet/home/rbandaru/ravi/finaletoolkit_figures/2B/tmp/"
pattern = "*.ctcf.intervals.wcov.bed"
file_paths = glob.glob(os.path.join(directory, pattern))

all_arrays = []
for file_path in tqdm(file_paths):
    fifth_column = []
    with open(file_path, 'r') as file:
        for line in file:
            fields = line.strip().split('\t')
            if len(fields) >= 5:
                fifth_column.append(float(fields[4]))
    fifth_column = np.array(fifth_column)
    total_sum = fifth_column.sum()
    if total_sum > 0:
        normalized_column = fifth_column
        all_arrays.append(normalized_column)

if len(all_arrays) == 0:
    raise ValueError("No valid arrays found.")

array_lengths = [len(arr) for arr in all_arrays]
if len(set(array_lengths)) > 1:
    raise ValueError(f"All arrays are not the same length. Lengths: {array_lengths}")

stacked_arrays = np.stack(all_arrays)
average_array = stacked_arrays.mean(axis=0)
print("Average array:", average_array)


100%|██████████| 8713/8713 [00:20<00:00, 420.87it/s]

Average array: [33.21219428 33.19669308 33.20277873 ... 33.31197612 33.32208061
 33.31484671]


In [21]:
np.save('ctcf.normalizedaverages.npy', average_array)